# HIGH-Throughput Virtual Screening of Human Kappa Opioid Receptor
## Author: **Babak Mamnoon**
## Docking Engine: GNINA


# ============================================================
# SECTION 1 — INSTALL REQUIRED PACKAGES
# ============================================================

In [ ]:
!pip -q install py3Dmol rdkit-pypi pandas numpy tqdm meeko

!apt-get -qq install openbabel


# ============================================================
# SECTION 2 — CHECK GPU
# ============================================================

In [ ]:
!nvidia-smi

# ============================================================
# SECTION 3 — CREATE WORKING DIRECTORY
# ============================================================

In [ ]:
import os

WORKDIR = "/content/KOR_HTVS"

os.makedirs(WORKDIR, exist_ok=True)

os.chdir(WORKDIR)

print("Current directory:", os.getcwd())

# ============================================================
# SECTION 4 — DOWNLOAD HUMAN KAPPA OPIOID RECEPTOR
# PDB ID: 4DJH
# ============================================================

In [ ]:
!wget -q https://files.rcsb.org/download/4DJH.pdb

print("Downloaded PDB structure.")


# ============================================================
# SECTION 5 — VISUALIZE RECEPTOR STRUCTURE
# ============================================================

In [ ]:
import py3Dmol

with open("4DJH.pdb", "r") as f:
    pdb_data = f.read()

view = py3Dmol.view(width=1000, height=700)

view.addModel(pdb_data, "pdb")

view.setStyle(
    {'chain':'A'},
    {'cartoon':{'color':'spectrum'}}
)

# Highlight co-crystallized ligand
view.setStyle(
    {'hetflag':True},
    {'stick':{'radius':0.2}}
)

view.zoomTo()

view.show()

# ============================================================
# SECTION 6 — RECEPTOR PREPARATION
# ============================================================

In [ ]:
# Remove waters
!grep "^ATOM" 4DJH.pdb > receptor_clean.pdb

# Add hydrogens and optimize
!obabel receptor_clean.pdb -O receptor_prepared.pdb -h

print("Receptor prepared successfully.")

# ============================================================
# SECTION 7 — EXTRACT CO-CRYSTALLIZED LIGAND
# ============================================================

In [ ]:
# Inspect heteroatoms first
!grep HETATM 4DJH.pdb | head

# NOTE:
# Replace "JDC" below with actual ligand residue if needed.

LIGAND_RESNAME = "JDC"

!grep {LIGAND_RESNAME} 4DJH.pdb > reference_ligand.pdb

print("Reference ligand extracted.")

# ============================================================
# SECTION 8 — VISUALIZE BINDING SITE
# ============================================================

In [ ]:
v = py3Dmol.view(width=1000, height=700)

v.addModel(open("receptor_clean.pdb").read(), "pdb")
v.setStyle({'cartoon':{'color':'white'}})

v.addModel(open("reference_ligand.pdb").read(), "pdb")
v.setStyle({'model':1},
           {'stick':{'colorscheme':'greenCarbon'}})

v.zoomTo({'model':1})

v.show()

# ============================================================
# SECTION 9 — INSTALL GNINA
# ============================================================

In [ ]:
!wget -q https://github.com/gnina/gnina/releases/download/v1.0.3/gnina

!chmod +x gnina

!./gnina --version

# ============================================================
# SECTION 10 — FETCH LIGAND LIBRARY FROM GITHUB
# ============================================================

In [ ]:
# Replace with your raw GitHub CSV URL
GITHUB_CSV_URL = "https://raw.githubusercontent.com/Babakmamnoon/High-Throughput-Virtual-Screening-of-Human-Kappa-Opioid-Receptor/main/ligands.csv"

!wget -O ligands.csv $GITHUB_CSV_URL

import pandas as pd

df = pd.read_csv("ligands.csv")

print(df.head())

# ============================================================
# SECTION 11 — EXPECTED CSV FORMAT
# ============================================================
"""
Expected CSV columns:

compound_id,smiles

Example:

compound_1,CCO
compound_2,CCN(CC)CC
"""

In [ ]:
import pandas as pd

# Load original ChEMBL dataset
chembl_df = pd.read_csv("ligands.csv")

print("Original dataset shape:")
print(chembl_df.shape)

print("\nColumns in dataset:")
print(chembl_df.columns.tolist())

# ============================================================
# Select Required Columns
# ============================================================

"""
Typical ChEMBL columns:

- molecule_chembl_id
- canonical_smiles

We rename them to:

- compound_id
- smiles
"""

clean_df = chembl_df[
    [
        "molecule_chembl_id",
        "canonical_smiles"
    ]
].copy()

# Rename columns
clean_df.columns = [
    "compound_id",
    "smiles"
]

# ============================================================
# Remove Missing Values
# ============================================================

clean_df = clean_df.dropna(subset=["smiles"])

# ============================================================
# Remove Duplicate SMILES
# ============================================================

clean_df = clean_df.drop_duplicates(
    subset=["smiles"]
).reset_index(drop=True)

print("\nClean dataset shape:")
print(clean_df.shape)

# ============================================================
# Save Clean Dataset
# ============================================================

clean_df.to_csv(
    "clean_ligands.csv",
    index=False
)

print("\nClean ligand file saved.")

# Replace df with cleaned dataset
df = clean_df

df.head()

# ============================================================
# SECTION 12 — LIGAND STANDARDIZATION
# ============================================================

In [ ]:
from rdkit import Chem
from rdkit.Chem import AllChem
from tqdm import tqdm

os.makedirs("ligands_sdf", exist_ok=True)

valid_compounds = []

for idx, row in tqdm(clean_df.iterrows(), total=len(clean_df)):

    compound_id = str(row["compound_id"])
    smiles = str(row["smiles"])

    try:
        mol = Chem.MolFromSmiles(smiles)

        if mol is None:
            continue

        # Add hydrogens
        mol = Chem.AddHs(mol)

        # Generate 3D coordinates
        AllChem.EmbedMolecule(
            mol,
            randomSeed=42
        )

        # MMFF optimization
        AllChem.MMFFOptimizeMolecule(mol)

        output_file = f"ligands_sdf/{compound_id}.sdf"

        writer = Chem.SDWriter(output_file)
        writer.write(mol)
        writer.close()

        valid_compounds.append(compound_id)

    except:
        continue

print(f"Valid ligands prepared: {len(valid_compounds)}")

# ============================================================
# SECTION 13 — CONVERT LIGANDS TO PDBQT-LIKE FORMAT
# ============================================================

In [ ]:
os.makedirs("ligands_prepared", exist_ok=True)

sdf_files = os.listdir("ligands_sdf")

for sdf in tqdm(sdf_files):

    input_path = f"ligands_sdf/{sdf}"

    output_path = f"ligands_prepared/{sdf.replace('.sdf','.pdbqt')}"

    !obabel {input_path} -O {output_path} -h

print("Ligands converted successfully.")

# ============================================================
# SECTION 14 — TEST SINGLE DOCKING
# ============================================================

In [ ]:
test_ligand = os.listdir("ligands_sdf")[0]

!./gnina \
-r receptor_prepared.pdb \
-l ligands_sdf/{test_ligand} \
--autobox_ligand reference_ligand.pdb \
--autobox_add 6 \
-o test_docked.sdf \
--exhaustiveness 8 \
--cnn_scoring rescore \
--seed 42

print("Single docking test completed.")

# ============================================================
# SECTION 15 — HIGH-THROUGHPUT VIRTUAL SCREENING
# ============================================================

In [ ]:
import glob
import subprocess

os.makedirs("docking_results", exist_ok=True)

ligand_files = glob.glob("ligands_sdf/*.sdf")

results = []

for ligand in tqdm(ligand_files):

    ligand_name = os.path.basename(ligand).replace(".sdf","")

    output_file = f"docking_results/{ligand_name}_docked.sdf"

    command = f"""
    ./gnina \
    -r receptor_prepared.pdb \
    -l {ligand} \
    --autobox_ligand reference_ligand.pdb \
    --autobox_add 6 \
    --exhaustiveness 8 \
    --cnn_scoring rescore \
    --seed 42 \
    -o {output_file}
    """

    subprocess.run(command, shell=True)

print("HTVS completed.")

# ============================================================
# SECTION 16 — EXTRACT GNINA SCORES
# ============================================================

In [ ]:
import re

docking_summary = []

result_files = glob.glob("docking_results/*_docked.sdf")

for file in result_files:

    ligand_name = os.path.basename(file).replace("_docked.sdf","")

    with open(file, "r") as f:
        content = f.read()

    affinity_match = re.search(
        r'> <minimizedAffinity>\s*\n([-.\d]+)',
        content
    )

    cnn_match = re.search(
        r'> <CNNaffinity>\s*\n([-.\d]+)',
        content
    )

    vina_score = affinity_match.group(1) if affinity_match else None
    cnn_score = cnn_match.group(1) if cnn_match else None

    docking_summary.append([
        ligand_name,
        vina_score,
        cnn_score
    ])

results_df = pd.DataFrame(
    docking_summary,
    columns=[
        "Ligand",
        "Vina_Affinity",
        "CNN_Affinity"
    ]
)

results_df = results_df.sort_values(
    by="CNN_Affinity",
    ascending=False
)

results_df.to_csv(
    "KOR_HTVS_Results.csv",
    index=False
)

results_df.head(20)

# ============================================================
# SECTION 17 — VISUALIZE TOP HIT
# ============================================================

In [ ]:
top_hit = results_df.iloc[0]["Ligand"]

top_hit_file = f"docking_results/{top_hit}_docked.sdf"

v = py3Dmol.view(width=1000, height=700)

v.addModel(open("receptor_clean.pdb").read(), "pdb")
v.setStyle({'cartoon':{'color':'white'}})

v.addModelsAsFrames(open(top_hit_file).read())

v.setStyle(
    {'model':1},
    {'stick':{'colorscheme':'cyanCarbon'}}
)

v.zoomTo()

v.show()

# ============================================================
# SECTION 18 — SAVE COMPLETE PROJECT
# ============================================================

In [ ]:
!zip -r KOR_HTVS_Project.zip \
receptor_prepared.pdb \
docking_results \
KOR_HTVS_Results.csv \
ligands_sdf

print("Project archived successfully.")

# ============================================================
# SECTION 19 — DOWNLOAD RESULTS
# ============================================================

In [ ]:
from google.colab import files

files.download("KOR_HTVS_Project.zip")

# ============================================================
# SECTION 20 — OPTIONAL: MOUNT GOOGLE DRIVE
# ============================================================

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

# Example save path
!cp KOR_HTVS_Project.zip /content/drive/MyDrive/

print("Results copied to Google Drive.")

# ============================================================
# END OF NOTEBOOK
# ============================================================